# `noaa_coops`: New Features

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GClunies/noaa_coops/blob/main/examples/noaa_coops_new_features_tutorial.ipynb)

`noaa_coops` is a Python wrapper around NOAA's CO-OPS APIs for tide, current, and water level data.

This notebook walks through what's new: improved pagination, `daily_max_min` support, derived products (DPAPI), and enhanced metadata access.

## Setup

This branch hasn't merged upstream yet, so `pip install noaa_coops` won't include these features. Run the cell below once — it installs this branch. `%pip` installs into whatever kernel is currently running this notebook, so pick any Python 3.10+ kernel and just run the cells top to bottom — no separate environment setup required.

Once this lands upstream, swap the install line for `%pip install -q noaa_coops`.

In [1]:
%pip install -q "noaa_coops @ git+https://github.com/staree14/noaa_coops_dev.git@gsoc-2026-final"
print("Setup complete — noaa_coops installed.")

/Users/gclunies/.cache/uv/builds-v0/.tmp75TEhj/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.
Setup complete — noaa_coops installed.


In [2]:
from noaa_coops import Station

## What's new, at a glance

- Improved pagination across NOAA's per-product date-range caps (no more incorrect chunking for some products' multi-year requests)
- `daily_max_min` product support, with `max_min_type` to filter to just `"max"` or `"min"`
- Derived products (DPAPI) — HTF flooding counts, extreme water levels, sea level trends and projections
- Cleaner, more reliable metadata (tide offsets, current bins, datums, `center_bin_1_dist` for current stations)

### Pagination that works correctly!

NOAA caps how much date range a single request can cover, and the cap varies by product (31 days for `water_level`, 365 for `hourly_height`, up to 10 years for `daily_mean`). Requesting a wider range used to mean chunking it into hardcoded blocks which weren't accurate for every product. Now `get_data()` does the right chunking, and prodcuts data is returned at a higher speed 


In [3]:
# A 3-year daily_mean range — past the 31-day cap most products carry, one call.
p = Station("9063038")  # Erie, Lake Erie
df= p.get_data(
    begin_date="20200630",
    end_date="20230630",
    datum="IGLD",
    time_zone="gmt",
    interval="hilo",
    product="daily_mean",
    units="english",
)

print(df.attrs.get("missing_blocks", []))  # empty = clean stitch across blocks
df

[]


,v,f
t,,
2020-06-30,574.357,"0,0"
2020-07-01,574.354,"0,0"
2020-07-02,574.406,"0,0"
2020-07-03,574.393,"0,0"
2020-07-04,574.337,"0,0"
...,...,...
2023-06-26,572.736,"0,0"
2023-06-27,572.999,"0,0"
2023-06-28,572.861,"0,0"


### `daily_max_min` support

Returns NOAA's daily extrema instead of the standard `v`/`s`/`f`/`q` shape — each day contributes a `max` row and a `min` row, distinguished by `record_type`. `interval` defaults to `"h"` if you don't set it, so 6-minute and hourly data never get mixed in the same DataFrame.

In [4]:
# Example 1: interval="6" pulls both max and min at 6-minute-derived resolution
p = Station("9491094")
df = p.get_data(begin_date="20170101", end_date="20170102",
                 datum="STND", product="daily_max_min",
                 interval=6, units="english", time_zone="gmt")
df

,record_type,value,pcComplete,flag
2017-01-01 09:42:00,max,9.951,100,0
2017-01-02 22:24:00,max,8.488,100,0
2017-01-01 23:54:00,min,8.425,100,0
2017-01-02 17:24:00,min,7.198,100,0


In [5]:
# Example 2: max_min_type="max" — only the daily highs, useful when you only care about peak levels (e.g. flood risk)
q = Station("9447130")
df2 = q.get_data(begin_date="20150101", end_date="20150110",
                  datum="MLLW", product="daily_max_min",
                  max_min_type="max", units="metric") # defaults to hourly
df2 

,record_type,value,pcComplete,flag
2015-01-01 21:00:00,max,3.392,100,0
2015-01-02 22:00:00,max,3.392,100,0
2015-01-03 13:00:00,max,3.318,100,0
2015-01-04 13:00:00,max,3.468,100,0
2015-01-05 14:00:00,max,3.682,100,0
2015-01-06 14:00:00,max,3.674,100,0
2015-01-07 15:00:00,max,3.654,100,0
2015-01-08 15:00:00,max,3.634,100,0
2015-01-09 16:00:00,max,3.696,100,0
2015-01-10 16:00:00,max,3.666,100,0


### Derived products via DPAPI

Some NOAA products (high tidal flooding counts/outlooks, extreme water levels, sea level trends and rise projections) live on a separate API (DPAPI) — now accessible through the same `Station` interface via `get_derived_product()`. Below are examples across the three families: HTF flooding, sea level trends/projections, and extreme water levels.

In [6]:
a = Station("9447130")
a.get_derived_product(product="htf_daily", start_date="20180101", end_date="20180630")

,day,maxWL,maxTime,majFlag,modFlag,minFlag,nanFlag,percent_completeness
0,01/01/2018,6.032,23,0,0,0,0,100.0
1,01/02/2018,6.000,14,0,0,0,0,100.0
2,01/03/2018,6.156,14,0,0,0,0,100.0
3,01/04/2018,6.334,15,0,0,0,0,100.0
4,01/05/2018,6.405,16,0,0,0,0,100.0
...,...,...,...,...,...,...,...,...
176,06/26/2018,5.572,11,0,0,0,0,100.0
177,06/27/2018,5.673,1,0,0,0,0,100.0
178,06/28/2018,5.794,2,0,0,0,0,100.0
179,06/29/2018,5.854,2,0,0,0,0,100.0


`htf_monthly` returns the daily flooding flags up into monthly counts by severity (`minCount`/`modCount`/`majCount`) — useful for flooding frequency trends.

In [7]:
# Eagle Point, TX — Gulf coast station with real minor-flooding counts in this window
htf = Station("8771013")
htf.get_derived_product(product="htf_monthly", start_date="20190601", end_date="20191231")


,stnId,stnName,lat,lon,year,month,majCount,modCount,minCount,nanCount,percent_completeness
0,8771013,"Eagle Point, Galveston Bay, TX",29.481306,-94.91725,2019,6,0,1,2,0,100.0
1,8771013,"Eagle Point, Galveston Bay, TX",29.481306,-94.91725,2019,7,0,0,0,2,93.6
2,8771013,"Eagle Point, Galveston Bay, TX",29.481306,-94.91725,2019,8,0,0,0,7,77.4
3,8771013,"Eagle Point, Galveston Bay, TX",29.481306,-94.91725,2019,9,0,0,15,0,100.0
4,8771013,"Eagle Point, Galveston Bay, TX",29.481306,-94.91725,2019,10,0,0,22,0,100.0
5,8771013,"Eagle Point, Galveston Bay, TX",29.481306,-94.91725,2019,11,0,0,0,0,100.0
6,8771013,"Eagle Point, Galveston Bay, TX",29.481306,-94.91725,2019,12,0,0,0,0,100.0


#### Sea level trends & projections

In [8]:
a = Station("9461380")
a.get_derived_product(product="sea_level_trends")

,stationId,stationName,affil,trendUnits,seasonalUnits,seasonalAverage,latitude,longitude,trendType,autoregressive,autoregressiveError,trend,trendError,y2000_offset,startDate,endDate
0,9461380,Adak Island,US,mm/yr,meters,-0.022,51.860639,-176.637583,SINGLE,0.3,0.03,-2.3,0.17,0.0,04/09/1957,12/15/2025


In [9]:
a = Station("9461380")
a.get_derived_product(product="sea_level_trends",detail="seasonal_cycle")

,month,seasonalVar,seasonalErr,stationId,stationName,affil,trendUnits,seasonalUnits,seasonalAverage,latitude,longitude,trendType,autoregressive,autoregressiveError,trend,trendError,y2000_offset,startDate,endDate
0,1,0.082,0.008,9461380,Adak Island,US,mm/yr,meters,-0.022,51.860639,-176.637583,SINGLE,0.3,0.03,-2.3,0.17,0.0,04/09/1957,12/15/2025
1,2,0.059,0.008,9461380,Adak Island,US,mm/yr,meters,-0.022,51.860639,-176.637583,SINGLE,0.3,0.03,-2.3,0.17,0.0,04/09/1957,12/15/2025
2,3,-0.017,0.008,9461380,Adak Island,US,mm/yr,meters,-0.022,51.860639,-176.637583,SINGLE,0.3,0.03,-2.3,0.17,0.0,04/09/1957,12/15/2025
3,4,-0.059,0.008,9461380,Adak Island,US,mm/yr,meters,-0.022,51.860639,-176.637583,SINGLE,0.3,0.03,-2.3,0.17,0.0,04/09/1957,12/15/2025
4,5,-0.044,0.008,9461380,Adak Island,US,mm/yr,meters,-0.022,51.860639,-176.637583,SINGLE,0.3,0.03,-2.3,0.17,0.0,04/09/1957,12/15/2025
5,6,-0.043,0.008,9461380,Adak Island,US,mm/yr,meters,-0.022,51.860639,-176.637583,SINGLE,0.3,0.03,-2.3,0.17,0.0,04/09/1957,12/15/2025
6,7,-0.054,0.008,9461380,Adak Island,US,mm/yr,meters,-0.022,51.860639,-176.637583,SINGLE,0.3,0.03,-2.3,0.17,0.0,04/09/1957,12/15/2025
7,8,-0.037,0.008,9461380,Adak Island,US,mm/yr,meters,-0.022,51.860639,-176.637583,SINGLE,0.3,0.03,-2.3,0.17,0.0,04/09/1957,12/15/2025
8,9,-0.011,0.008,9461380,Adak Island,US,mm/yr,meters,-0.022,51.860639,-176.637583,SINGLE,0.3,0.03,-2.3,0.17,0.0,04/09/1957,12/15/2025
9,10,0.004,0.008,9461380,Adak Island,US,mm/yr,meters,-0.022,51.860639,-176.637583,SINGLE,0.3,0.03,-2.3,0.17,0.0,04/09/1957,12/15/2025


`slr_projections` gives forward-looking rise estimates instead of the trend above — filter by `scenario` (`"low"` through `"extreme"`), `projection_year`, and `report_year`.

In [10]:
# Adak, AK — "high" scenario projection for 2050, from NOAA's 2022 report
adak = Station("9461380")
slr = adak.get_derived_product(
    product="slr_projections", projection_year=2050, report_year=2022, scenario="high"
)
slr[["stationName", "scenario", "projectionYear", "reportYear", "projectionRsl", "projectionCiLow", "projectionCiHigh"]]

,stationName,scenario,projectionYear,reportYear,projectionRsl,projectionCiLow,projectionCiHigh
0,ADAK SWEEPER_COVE,High,2050,2022,23.0,12.0,40.0


#### RFA extreme water levels

Returns exploded, repeated station-attribute rows so that every return-period row still carries its own station identity — convenient for concatenating results across multiple stations without a separate join.

In [11]:
b= Station("1611347")
b.get_derived_product(product="rfa_extreme_water_levels")

,rl,value,ci_low,ci_high,stationId,name,lat,lon,tidalEpoch,gridNum,...,validDays,totalDays,percentValidDays,probabilitiesUnit,long_term_flag,short_term_flag,localUIndex_value,localUIndex_unit,localUTrend_value,localUTrend_unit
0,0.1yr,0.23,0.18,0.28,1611347,"PORT ALLEN, HANAPEPE BAY, KAUAI ISLAND",21.9033,-159.5920,1983-2001,39509,...,2896,2905,99.7,m,False,True,0.233,m,2.1,mm
1,0.2yr,0.27,0.20,0.33,1611347,"PORT ALLEN, HANAPEPE BAY, KAUAI ISLAND",21.9033,-159.5920,1983-2001,39509,...,2896,2905,99.7,m,False,True,0.233,m,2.1,mm
2,0.3yr,0.29,0.22,0.36,1611347,"PORT ALLEN, HANAPEPE BAY, KAUAI ISLAND",21.9033,-159.5920,1983-2001,39509,...,2896,2905,99.7,m,False,True,0.233,m,2.1,mm
3,0.4yr,0.30,0.23,0.38,1611347,"PORT ALLEN, HANAPEPE BAY, KAUAI ISLAND",21.9033,-159.5920,1983-2001,39509,...,2896,2905,99.7,m,False,True,0.233,m,2.1,mm
4,0.5yr,0.32,0.24,0.39,1611347,"PORT ALLEN, HANAPEPE BAY, KAUAI ISLAND",21.9033,-159.5920,1983-2001,39509,...,2896,2905,99.7,m,False,True,0.233,m,2.1,mm
5,0.6yr,0.32,0.24,0.41,1611347,"PORT ALLEN, HANAPEPE BAY, KAUAI ISLAND",21.9033,-159.5920,1983-2001,39509,...,2896,2905,99.7,m,False,True,0.233,m,2.1,mm
6,0.7yr,0.33,0.25,0.42,1611347,"PORT ALLEN, HANAPEPE BAY, KAUAI ISLAND",21.9033,-159.5920,1983-2001,39509,...,2896,2905,99.7,m,False,True,0.233,m,2.1,mm
7,0.8yr,0.34,0.26,0.43,1611347,"PORT ALLEN, HANAPEPE BAY, KAUAI ISLAND",21.9033,-159.5920,1983-2001,39509,...,2896,2905,99.7,m,False,True,0.233,m,2.1,mm
8,0.9yr,0.35,0.26,0.43,1611347,"PORT ALLEN, HANAPEPE BAY, KAUAI ISLAND",21.9033,-159.5920,1983-2001,39509,...,2896,2905,99.7,m,False,True,0.233,m,2.1,mm
9,1yr,0.35,0.26,0.44,1611347,"PORT ALLEN, HANAPEPE BAY, KAUAI ISLAND",21.9033,-159.5920,1983-2001,39509,...,2896,2905,99.7,m,False,True,0.233,m,2.1,mm


`extreme_water_levels` is the related single-station endpoint behind the RFA return-period
statistics above. Filter with `level_type="high"` or `"low"`.

In [12]:
extremes = a.get_derived_product(product="extreme_water_levels", units="metric", level_type="low")
extremes[["type", "date", "status"]] 
# uncomment this next line to see full DataFrame
# extremes 

,type,date,status
0,GEV_LO,12/11/1946,NO
1,GEV_LO,05/31/1950,YES
2,GEV_LO,06/01/1950,YES
3,GEV_LO,11/12/1950,YES
4,GEV_LO,12/03/1967,YES
5,GEV_LO,06/13/1991,YES
6,GEV_LO,11/26/2003,YES
7,GEV_LO,02/25/2006,YES
8,GEV_LO,06/15/2007,YES
9,GEV_LO,12/13/2008,YES


### Cleaner metadata access

Fields like tide prediction offsets and current bins used to be inconsistent across station types. Access is now consistent regardless of station quirks.

In [13]:
# Station with multiple current bins
u = Station("ACT0921")
u.current_pred_offsets_by_bin

{1: {'id': 'ACT0921',
  'refStationId': 'BOS1111',
  'refStationBin': 14,
  'meanFloodDir': 239.0,
  'meanEbbDir': 63.0,
  'mfcTimeAdjMin': 60,
  'sbeTimeAdjMin': 37,
  'mecTimeAdjMin': -13,
  'sbfTimeAdjMin': 44,
  'mfcAmpAdj': 0.4,
  'mecAmpAdj': 0.4,
  'self': 'https://api.tidesandcurrents.noaa.gov/mdapi/prod/webapi/stations/ACT0921_1/currentpredictionoffsets.json'},
 2: {'id': 'ACT0921',
  'refStationId': 'BOS1111',
  'refStationBin': 14,
  'meanFloodDir': 224.0,
  'meanEbbDir': 48.0,
  'mfcTimeAdjMin': 39,
  'sbeTimeAdjMin': 52,
  'mecTimeAdjMin': 17,
  'sbfTimeAdjMin': 25,
  'mfcAmpAdj': 0.4,
  'mecAmpAdj': 0.3,
  'self': 'https://api.tidesandcurrents.noaa.gov/mdapi/prod/webapi/stations/ACT0921_2/currentpredictionoffsets.json'},
 3: {'id': 'ACT0921',
  'refStationId': 'BOS1111',
  'refStationBin': 14,
  'meanFloodDir': 271.0,
  'meanEbbDir': 35.0,
  'mfcTimeAdjMin': -54,
  'sbeTimeAdjMin': 9,
  'mecTimeAdjMin': -43,
  'sbfTimeAdjMin': -46,
  'mfcAmpAdj': 0.3,
  'mecAmpAdj': 0.3,


In [14]:
# Subordinate station, null datums but has tidePredOffsets
s = Station("8557863")
s.tide_pred_offsets

{'refStationId': '8570280',
 'type': 'S',
 'heightOffsetHighTide': 1.13,
 'heightOffsetLowTide': 1.33,
 'timeOffsetHighTide': 15,
 'timeOffsetLowTide': 8,
 'heightAdjustedType': 'R',
 'self': None}

In [15]:
# Subordinate station, with datums and tidePredOffsets
t = Station("8720135")  
t.tide_pred_offsets

{'refStationId': '8720030',
 'type': 'S',
 'heightOffsetHighTide': 0.86,
 'heightOffsetLowTide': 1.0,
 'timeOffsetHighTide': -18,
 'timeOffsetLowTide': 41,
 'heightAdjustedType': 'R',
 'self': None}

In [16]:
# Harmonic station, has datums and no tidePredOffsets
q = Station("8570280")
q.datums

{'accepted': 'Apr 17 2003',
 'superseded': '',
 'epoch': '1983-2001',
 'units': 'meters',
 'OrthometricDatum': 'NAVD88',
 'datums': [{'name': 'STND', 'description': 'Station Datum', 'value': 0.0},
  {'name': 'MHHW', 'description': 'Mean Higher-High Water', 'value': 2.473},
  {'name': 'MHW', 'description': 'Mean High Water', 'value': 2.359},
  {'name': 'DTL', 'description': 'Mean Diurnal Tide Level', 'value': 1.879},
  {'name': 'MTL', 'description': 'Mean Tide Level', 'value': 1.847},
  {'name': 'MSL', 'description': 'Mean Sea Level', 'value': 1.853},
  {'name': 'MLW', 'description': 'Mean Low Water', 'value': 1.334},
  {'name': 'MLLW', 'description': 'Mean Lower-Low Water', 'value': 1.286},
  {'name': 'GT', 'description': 'Great Diurnal Range', 'value': 1.187},
  {'name': 'MN', 'description': 'Mean Range of Tide', 'value': 1.024},
  {'name': 'DHQ',
   'description': 'Mean Diurnal High Water Inequality',
   'value': 0.114},
  {'name': 'DLQ',
   'description': 'Mean Diurnal Low Water Ine

### Where to go next

- Full docs: in README.md
- Deferred to a later phase: `peakwaterlevels`, `extremewaterlevels` sub-endpoint, `htf_outlook`, bounding-box queries 